# Create STAC for dataproduct "vern"

Load YAML config and run feature_class_layers.

In [59]:
from pathlib import Path
from src.yaml_config import load_map_service_config, feature_class_layers

from datetime import datetime, timezone
import pystac

In [60]:
config_path = Path("./config/vern_arcgis_map_service_definition.yml").resolve()
config, config_map, config_layers = load_map_service_config(config_path)

print(f"Loaded: {config_path}")
print(f"Map name: {config_map.get('name')}")
print(f"Total layer entries: {len(config_layers)}")
print(f"Feature class layers: {len(fc_layers)}")

Loaded: C:\Users\WILACA\git\kartai\skygeo\src\stac_structure\config\vern_arcgis_map_service_definition.yml
Map name: vern
Total layer entries: 8
Feature class layers: 6


In [61]:
fc_layers = feature_class_layers(config_layers)
print(f"Feature class layers: {len(fc_layers)}")

for layer_name, layer_cfg in fc_layers:
    service_id = layer_cfg.get("service_id")
    print(f"- service_id={service_id}, layer={layer_name}")

Feature class layers: 6
- service_id=0, layer=naturvern_omraade
- service_id=1, layer=naturvern_klasser_omraade
- service_id=2, layer=naturvern_teiggrensepunkt
- service_id=3, layer=naturvern_grense
- service_id=4, layer=foreslaatt_naturvern_omraade
- service_id=5, layer=foreslaatt_naturvern_grense


In [62]:
# Inspect the first feature class layer config
first_name, first_cfg = fc_layers[0]
print(first_name)
first_cfg

naturvern_omraade


{'type': 'feature_class',
 'source': {'feature_dataset': 'sde_feature_dataset',
  'dataset': 'sde_dataset'},
 'service_id': 0,
 'visible': True,
 'display_field': 'naturvern_id',
 'cols': ['naturvern_id',
  'cdda_id',
  'navn',
  'offisielt_navn',
  'faktaark',
  'verneform',
  'verneforskrift',
  'vernedato',
  'foerstegang_vernet',
  'verneplan',
  'kommune',
  'forvaltningsmyndighet',
  'forvaltningsmyndighet_type',
  'iucn',
  'revisjon',
  'truet_vurdering',
  'major_ecosystem_type',
  'verneform_aggregert',
  'objekttype',
  'objectid',
  'globalid',
  'shape',
  'st_area(shape)',
  'st_perimeter(shape)']}

## Create STAC item

Get collection metadata from YAML

In [63]:
# Fixed requirements from your decision
item_id = "vern_norge_20260615"
generation_time = datetime.now(timezone.utc)

collection_id = "vern"
description = config_map.get("metadata", {}).get("description", "Vernedata")
license_name = config_map.get("metadata", {}).get("license", "proprietary")

service_name = str(config_map.get("name", "vern")).strip()
base_url = "https://vsepublicstorage.blob.core.windows.net/vse-public/stac/miljodir-stac/collections"
lyrx_asset_base = f"{base_url}/vern/assets/lyrx"

bbox = [27250.06038219, 6579917.07024015, 1167250.06038219, 7939917.07024015]
geom = {
    "type": "Polygon",
    "coordinates": [[
        [bbox[0], bbox[1]],
        [bbox[0], bbox[3]],
        [bbox[2], bbox[3]],
        [bbox[2], bbox[1]],
        [bbox[0], bbox[1]],
    ]],
}

├── vern (collection)
│   ├── vern_norge_20260615 (item)
│   │   ├── vern_0_naturvern_omraade_lyrx
│   │   ├── vern_1
│   │   └── 
│   ├── vern_norge_20260615 (item)


In [64]:
# One item with many LYRX assets
item = pystac.Item(
    id=item_id,
    geometry=geom,
    bbox=bbox,
    datetime=generation_time,
    properties={
        "generation_time": generation_time.isoformat(),
        "asset_count_expected": len(fc_layers),
    },
)

for layer_name, layer_cfg in fc_layers:
    service_id = layer_cfg.get("service_id")
    lyrx_filename = f"{service_name}_{service_id}_{layer_name}.lyrx"
    lyrx_href = f"{lyrx_asset_base}/{lyrx_filename}"

    asset_key = f"{layer_name}_lyrx"
    item.add_asset(
        asset_key,
        pystac.Asset(
            href=lyrx_href,
            media_type="application/octet-stream",
            roles=["style", "metadata"],
            title=f"LYRX style for {layer_name}",
        ),
    )

print(f"Item id: {item.id}")
print(f"Assets on item: {len(item.assets)}")

Item id: vern_norge_20260615
Assets on item: 6


In [65]:
collection = pystac.Collection(
    id=collection_id,
    description=description,
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([bbox]),
        temporal=pystac.TemporalExtent([[generation_time, generation_time]]),
    ),
    license=license_name,
)

collection.add_item(item)

print(f"Collection id: {collection.id}")
print(f"Items in collection: {len(list(collection.get_items()))}")

Collection id: vern
Items in collection: 1


In [66]:
catalog_root = Path("collections")
collection_dir = catalog_root / "vern"
collection_dir.mkdir(parents=True, exist_ok=True)

catalog = pystac.Catalog(
    id="miljodir",
    description="STAC catalog for Miljodir dataproducts",
)
catalog.add_child(collection)

catalog.normalize_hrefs(str(catalog_root))
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

print(f"Saved catalog: {catalog_root / 'catalog.json'}")
print(f"Saved collection: {collection_dir / 'collection.json'}")
print(f"Saved item folder: {collection_dir / item.id}")

Saved catalog: collections\catalog.json
Saved collection: collections\vern\collection.json
Saved item folder: collections\vern\vern_norge_20260615


### Upload STAC JSON source and LYRX assets to Azure

- azcopy to `collection` to blob storage
- azcopy lyrx assest to `collections/vern/assets/lyrx`


### Demo: Access STAC and download assets

Open the remote STAC, find dataproduct `vern`, locate item `vern_norge_20260615`, and download all item assets to a local folder.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import urlopen
from urllib.error import HTTPError
import shutil
import pystac

# Try catalog path with /stac first, then fallback to path without /stac
CATALOG_CANDIDATES = [
    "https://vsepublicstorage.blob.core.windows.net/vse-public/stac/miljodir-stac/collections/catalog.json",
    "https://vsepublicstorage.blob.core.windows.net/vse-public/miljodir-stac/collections/catalog.json",
]

COLLECTION_ID = "vern"
ITEM_ID = "vern_norge_20260615"
DOWNLOAD_DIR = Path("downloads/vern_norge_20260615")

catalog = None
catalog_url_used = None
for candidate in CATALOG_CANDIDATES:
    try:
        catalog = pystac.Catalog.from_file(candidate)
        catalog_url_used = candidate
        break
    except Exception:
        pass

if catalog is None:
    raise RuntimeError(
        "Could not open STAC catalog from candidate URLs. "
        "Check that catalog.json is uploaded and publicly accessible."
    )

print(f"Catalog loaded: {catalog_url_used}")

collection = catalog.get_child(COLLECTION_ID, recursive=True)
if collection is None:
    raise RuntimeError(f"Collection not found: {COLLECTION_ID}")

print(f"Collection found: {collection.id}")

item = collection.get_item(ITEM_ID)
if item is None:
    available = [it.id for it in collection.get_items()]
    raise RuntimeError(
        f"Item not found: {ITEM_ID}. Available items: {available}"
    )

print(f"Item found: {item.id}")
print(f"Asset count: {len(item.assets)}")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
downloaded = []
missing = []

def candidate_asset_urls(href: str, item_id: str, filename: str) -> list[str]:
    urls = [href]

    # Fallback 1: if blob path has /stac/ but files were uploaded without it
    if "/stac/" in href:
        urls.append(href.replace("/stac/", "/", 1))

    # Fallback 2: if files were uploaded under item folder instead of assets/lyrx
    marker = f"/assets/lyrx/{filename}"
    if marker in href:
        urls.append(href.replace(marker, f"/{item_id}/{filename}"))

    # Fallback 3: combine both transformations
    if "/stac/" in href and marker in href:
        combined = href.replace("/stac/", "/", 1).replace(
            marker, f"/{item_id}/{filename}"
        )
        urls.append(combined)

    # De-duplicate while preserving order
    unique = []
    for url in urls:
        if url not in unique:
            unique.append(url)
    return unique

for key, asset in item.assets.items():
    href = asset.href
    if not href:
        print(f"Skip {key}: empty href")
        continue

    filename = Path(urlparse(href).path).name or f"{key}.bin"
    out_path = DOWNLOAD_DIR / filename

    success = False
    attempts = candidate_asset_urls(href, ITEM_ID, filename)

    for candidate_url in attempts:
        try:
            with urlopen(candidate_url, timeout=60) as src, open(out_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
            downloaded.append(out_path)
            print(f"Downloaded {key}: {out_path}")
            if candidate_url != href:
                print(f"  Used fallback URL: {candidate_url}")
            success = True
            break
        except HTTPError as err:
            if err.code == 404:
                continue
            raise

    if not success:
        missing.append((key, href))
        print(f"Missing {key}: {href}")

print()
print(f"Done. Downloaded {len(downloaded)} assets to: {DOWNLOAD_DIR.resolve()}")
if missing:
    print(f"Assets still missing: {len(missing)}")
    for key, href in missing:
        print(f"- {key}: {href}")

Catalog loaded: https://vsepublicstorage.blob.core.windows.net/vse-public/miljodir-stac/collections/catalog.json
Collection found: vern
Item found: vern_norge_20260615
Asset count: 6


HTTPError: HTTP Error 404: The specified blob does not exist.